# Recomendador de Artículos Similares por Perfil de Tópicos

Este notebook carga el modelo híbrido de BERTopic y las distribuciones de probabilidad de los tópicos para buscar los 10 artículos más cercanos temáticamente a un `doc_id` dado, utilizando la distancia coseno.

In [13]:
# 1. Parche de compatibilidad obligatorio para BERTopic
import coverage.types as ct
for name, fallback in {
    'Tracer': getattr(ct, 'TTracer', object),
    'TShouldTraceFn': object,
    'TShouldStartContextFn': object,
    'TFileDisposition': object,
}.items():
    if not hasattr(ct, name):
        setattr(ct, name, fallback)

# 2. Imports generales
import pandas as pd
import numpy as np
from pathlib import Path
from bertopic import BERTopic
from sklearn.metrics.pairwise import cosine_similarity
from umap import UMAP
import plotly.graph_objects as go
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings("ignore")

# 3. Configuración de Rutas
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

from bertopic.backend._utils import select_backend

INPUT_DIR = PROJECT_ROOT / "outputs" / "bertopic_acotado"

# 4. Cargar Parquet final y embeddings ya calculados
df = pd.read_parquet(INPUT_DIR / "df_completo.parquet")
if "doc_id" not in df.columns:
    df["doc_id"] = df.index

embeddings_naturales = np.load(INPUT_DIR / "embeddings_naturales.npy")

# 5. Cargar modelo final
topic_model = BERTopic.load(str(INPUT_DIR / "modelo_completo"))
raw_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
topic_model.embedding_model = select_backend(raw_model)

print(f"Corpus cargado: {len(df):,} documentos listos para analizar.")

2026-10-15 23:14:55,639 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7653.77it/s]


Corpus cargado: 3,556 documentos listos para analizar.


In [14]:
# ==============================================================================
# CARGA DEL MODELO Y DISTRIBUCIONES DE PROBABILIDAD DE TÓPICOS
# ==============================================================================



# ==============================================================================
# DISTRIBUCIONES DE PROBABILIDAD DE TÓPICOS
# ==============================================================================
print("Calculando distribuciones continuas de tópicos para todos los documentos...")
textos_docs = df["document_text_final"].fillna("").astype(str).tolist()
doc_topic_dist, _ = topic_model.approximate_distribution(textos_docs, use_embedding_model=False)

print(f"Matriz de distribución generada. Shape: {doc_topic_dist.shape}")

Calculando distribuciones continuas de tópicos para todos los documentos...
Matriz de distribución generada. Shape: (3556, 23)


In [15]:
# ==============================================================================
# FUNCIÓN DE SIMILITUD Y AUDITORÍA DE ETIQUETAS (CON TAGLINE)
# ==============================================================================

def recomendar_similares_por_topico(target_doc_id, top_n=10):
    """
    Toma un doc_id, busca su perfil de tópicos y devuelve los top_n artículos 
    con mayor similitud coseno, analizando el estado de sus etiquetas y mostrando la bajada.
    """
    if target_doc_id not in df["doc_id"].values:
        return f"Error: doc_id {target_doc_id} no encontrado en el corpus."
    
    # 1. Extraer vector de distribución temática del documento objetivo
    target_vector = doc_topic_dist[target_doc_id].reshape(1, -1)
    
    # 2. Calcular Similitud Coseno contra todo el corpus
    similitudes = cosine_similarity(target_vector, doc_topic_dist).flatten()
    
    # 3. Ordenar índices de mayor a menor
    indices_similares = similitudes.argsort()[::-1][:top_n + 1]
    
    # Excluir el documento objetivo de la lista de resultados
    indices_similares = [idx for idx in indices_similares if idx != target_doc_id][:top_n]
    
    # 4. Construir tabla de resultados con diagnóstico de etiquetas y tagline
    resultados = []
    for rank, idx in enumerate(indices_similares, start=1):
        row = df.iloc[idx]
        
        # Validar existencia de la etiqueta
        etiqueta_raw = row.get("post_tag", np.nan)
        tiene_etiqueta = pd.notna(etiqueta_raw) and str(etiqueta_raw).strip() != ""
        
        # Limpiar tagline
        tagline_raw = str(row.get("tagline", "")).strip()
        tagline_limpio = tagline_raw if tagline_raw and tagline_raw.lower() != "nan" else "Sin bajada"
        
        resultados.append({
            "Rank": rank,
            "doc_id": row["doc_id"],
            "Similitud_Coseno": round(float(similitudes[idx]), 4),
            "Título": row["title_text"],
            "Tagline": tagline_limpio,
            #"Tiene_Etiqueta": tiene_etiqueta,
            "Etiquetas": str(etiqueta_raw).strip() if tiene_etiqueta else "Sin etiquetas"
        })
        
    df_resultados = pd.DataFrame(resultados)
    
    # Información del documento consultado
    doc_origen = df.iloc[target_doc_id]
    
    etiqueta_origen = doc_origen.get("post_tag", np.nan)
    origen_tiene_tag = pd.notna(etiqueta_origen) and str(etiqueta_origen).strip() != ""
    
    tagline_origen_raw = str(doc_origen.get("tagline", "")).strip()
    tagline_origen = tagline_origen_raw if tagline_origen_raw and tagline_origen_raw.lower() != "nan" else "Sin bajada"
    
    print("=" * 100)
    print(f"[ID: {target_doc_id}]")
    print(f"Título:  {doc_origen['title_text']}")
    print(f"Tagline: {tagline_origen}")
    print(f"Tags:    {etiqueta_origen if origen_tiene_tag else 'Sin etiquetas'}")
    print("=" * 100)
    print(f"\nTOP {top_n} NOTAS MÁS SIMILARES POR PERFIL DE TÓPICOS:\n")
    
    return df_resultados

In [16]:
# ==============================================================================
# 2. CÁLCULO SILENCIOSO DE LAS DOS DIMENSIONES (SEMÁNTICA Y TEMÁTICA)
# ==============================================================================

# A. Espacio Semántico (Embeddings de texto puro original)
# A. Espacio Semántico (Proyección 2D directa de embeddings_naturales)
if "umap_sem_x" not in df.columns:
    print("Reduciendo Espacio Semántico a 2D...")
    reducer_sem = UMAP(n_neighbors=15, n_components=2, metric='cosine', random_state=42)
    coords_sem = reducer_sem.fit_transform(embeddings_naturales)
    df["umap_sem_x"] = coords_sem[:, 0]
    df["umap_sem_y"] = coords_sem[:, 1]

# B. Espacio Temático (Probabilidades de Tópicos BERTopic)
if "umap_tem_x" not in df.columns:
    print("Preparando Espacio Temático (Distribución BERTopic)...")
    reducer_tem = UMAP(n_neighbors=15, n_components=2, metric='cosine', random_state=42)
    coords_tem = reducer_tem.fit_transform(doc_topic_dist)
    df["umap_tem_x"] = coords_tem[:, 0]
    df["umap_tem_y"] = coords_tem[:, 1]


Reduciendo Espacio Semántico a 2D...
Preparando Espacio Temático (Distribución BERTopic)...


In [17]:
import plotly.express as px
from umap import UMAP
import plotly.graph_objects as go
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# ==============================================================================
# 1. PREPARAR PALABRAS CLAVE DE TÓPICOS PARA LOS COLORES
# ==============================================================================
# 1. Obtener los IDs válidos excluyendo outliers (-1)
# 1. Obtener todos los IDs oficiales (0 al 21 + 100)
topic_info = topic_model.get_topic_info()
all_topics = sorted(df["topic"].unique())

# 2. Diccionario de nombres oficiales con sus keywords
topic_words_dict = {}
for t_id in all_topics:
    if t_id == 100:
        topic_words_dict[t_id] = "100_outliers_miscelaneo"
    else:
        words = [w for w, _ in topic_model.get_topic(t_id)[:3]]
        topic_words_dict[t_id] = f"{t_id}_" + "_".join(words)

# 3. Mapear y formatear etiquetas
df["topic_name"] = df["topic"].map(topic_words_dict)
df["topic_label"] = df["topic_name"].fillna("Sin tópico")

if "tagline" in df.columns:
    df["tagline"] = df["tagline"].fillna("").astype(str).replace(["nan", "None"], "")
else:
    df["tagline"] = ""

# 4. Orden numérico estricto para la leyenda
categorias_ordenadas = [topic_words_dict[t] for t in all_topics]
df["topic_label"] = pd.Categorical(df["topic_label"], categories=categorias_ordenadas, ordered=True)

# ==============================================================================
# 2. FUNCIÓN DE VISUALIZACIÓN MEJORADA (Colores + Formas)
# ==============================================================================
def visualizar_constelacion_color(target_doc_id, df_similares, col_x, col_y, titulo_grafico):
    recomendados_ids = df_similares["doc_id"].tolist()
    
    # Dividir los datos
    df_resto = df[~df["doc_id"].isin([target_doc_id] + recomendados_ids)]
    df_rec = df[df["doc_id"].isin(recomendados_ids)].copy()
    df_rec["Rank"] = df_rec["doc_id"].map(df_similares.set_index("doc_id")["Rank"])
    df_target = df[df["doc_id"] == target_doc_id]
    
    # Capa 1: Fondo coloreado por Tópicos (usando Plotly Express para gestionar la paleta)
    fig = px.scatter(
        df_resto, x=col_x, y=col_y, color='topic_label',
        color_discrete_sequence=px.colors.qualitative.Alphabet,
        custom_data=['topic_label', 'title_text', 'tagline']
    )
    
    # Hacer los puntos del fondo más chicos y transparentes
    fig.update_traces(
        marker=dict(size=6, opacity=0.5),
        hovertemplate="<b>%{customdata[0]}</b><br><b>%{customdata[1]}</b><br><i>%{customdata[2]}</i><extra></extra>"
    )
    
    # Capa 2: Las Recomendaciones (Círculos)
    fig.add_trace(go.Scatter(
        x=df_rec[col_x], y=df_rec[col_y], mode="markers", name="Top 10 Recomendados",
        marker=dict(symbol="circle", size=16, line=dict(width=3, color="#1f77b4")),
        hovertemplate="<b>Recomendación #%{customdata[1]}</b><br><b>Tópico:</b> %{customdata[2]}<br><b>%{customdata[0]}</b><br><i>%{customdata[3]}</i><extra></extra>", 
        customdata=df_rec[["title_text", "Rank", "topic_label", "tagline"]]
    ))
    
    # Capa 3: El artículo consultado / Origen (Diamante Rojo)
    fig.add_trace(go.Scatter(
        x=df_target[col_x], y=df_target[col_y], mode="markers", name="Artículo Elegido",
        marker=dict(color="#d62728", size=20, symbol="diamond", line=dict(width=2, color="black")),
        hovertemplate="<b>ARTÍCULO ORIGINAL</b><br><b>Tópico:</b> %{customdata[1]}<br><b>%{customdata[0]}</b><br><i>%{customdata[2]}</i><extra></extra>", 
        customdata=df_target[["title_text", "topic_label", "tagline"]]
    ))
    
    fig.update_layout(
        title=f"<b>{titulo_grafico}</b>",
        plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='#fff', height=800,
        hoverlabel=dict(bgcolor="white", font_size=14),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title=""),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title=""),
        legend=dict(title="<b>Referencias y Tópicos</b>", yanchor="top", y=1, xanchor="left", x=1.02)
    )
    return fig

In [18]:
import plotly.graph_objects as go
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

def visualizar_dispersion_temporal_jerarquica(target_doc_id, top_n=100):
    # 1. Calcular similitudes contra todo el corpus
    target_vector = doc_topic_dist[target_doc_id].reshape(1, -1)
    similitudes = cosine_similarity(target_vector, doc_topic_dist).flatten()
    
    # 2. Armar dataframe temporal
    df_temp = df.copy()
    df_temp["Similitud"] = similitudes
    if not pd.api.types.is_datetime64_any_dtype(df_temp["post_date"]):
        df_temp["post_date"] = pd.to_datetime(df_temp["post_date"], errors="coerce")
    
    # 3. Separar los tres grupos de datos
    df_origen = df_temp[df_temp["doc_id"] == target_doc_id].copy()
    
    # Excluimos el original y rankeamos el resto
    df_recs = df_temp[df_temp["doc_id"] != target_doc_id].sort_values("Similitud", ascending=False).head(top_n).copy()
    df_recs["Rank"] = range(1, top_n + 1)
    
    # Cortamos el Top 10 y el resto (11 al 100)
    df_top10 = df_recs[df_recs["Rank"] <= 10]
    df_resto = df_recs[df_recs["Rank"] > 10]
    
    # 4. Construcción del gráfico por capas
    fig = go.Figure()
    
    # Capa A: Rank 11 al 100 (Fondo translúcido)
    fig.add_trace(go.Scatter(
        x=df_resto["post_date"], 
        y=df_resto["Similitud"],
        mode="markers",
        name="Rank 11-100",
        marker=dict(color="#ab63fa", size=12, opacity=0.4, line=dict(width=1, color="black")),
        customdata=df_resto[["title_text", "Rank", "topic_label"]],
        hovertemplate=(
            "<b>Rank #%{customdata[1]}</b><br>"
            "<b>Similitud:</b> %{y:.3f}<br>"
            "<b>Fecha:</b> %{x|%Y-%m-%d}<br>"
            "<b>Tópico:</b> %{customdata[2]}<br>"
            "<b>Título:</b> %{customdata[0]}<extra></extra>"
        )
    ))
    
    # Capa B: Top 10 Recomendados (Estrellas doradas)
    fig.add_trace(go.Scatter(
        x=df_top10["post_date"], 
        y=df_top10["Similitud"],
        mode="markers",
        name="Top 10 Recomendados",
        marker=dict(symbol="circle", color="#FFD700", size=18, line=dict(width=1.5, color="black")),
        customdata=df_top10[["title_text", "Rank", "topic_label"]],
        hovertemplate=(
            "<b>Rank #%{customdata[1]}</b><br>"
            "<b>Similitud:</b> %{y:.3f}<br>"
            "<b>Fecha:</b> %{x|%Y-%m-%d}<br>"
            "<b>Tópico:</b> %{customdata[2]}<br>"
            "<b>Título:</b> %{customdata[0]}<extra></extra>"
        )
    ))
    
    # Capa C: Artículo Original (Diamante rojo)
    fecha_origen = df_origen["post_date"].iloc[0]
    fig.add_trace(go.Scatter(
        x=df_origen["post_date"], 
        y=df_origen["Similitud"],
        mode="markers",
        name="Artículo Consultado",
        marker=dict(symbol="diamond", color="#d62728", size=20, line=dict(width=2, color="black")),
        customdata=df_origen[["title_text", "topic_label"]],
        hovertemplate=(
            "<b>ARTÍCULO ORIGINAL</b><br>"
            "<b>Fecha:</b> %{x|%Y-%m-%d}<br>"
            "<b>Tópico:</b> %{customdata[1]}<br>"
            "<b>Título:</b> %{customdata[0]}<extra></extra>"
        )
    ))
    
    # Marcador temporal (Línea roja discontinua)
    fig.add_vline(
        x=fecha_origen, line_dash="dash", line_color="red", 
        annotation_text="← Publicación Artículo Original", annotation_position="top right"
    )
    
    # Formato general
    fig.update_layout(
        title=f"<b>Articulos respecto de esas notas: Top {top_n} notas similares al ID {target_doc_id}</b>",
        xaxis_title="Fecha de Publicación",
        yaxis_title="Score de Similitud Coseno",
        plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='#fff',
        yaxis=dict(showgrid=True, gridcolor="#ebebeb"),
        xaxis=dict(showgrid=True, gridcolor="#ebebeb"),
        hoverlabel=dict(bgcolor="white", font_size=13),
        legend=dict(title="<b>Referencias</b>", yanchor="top", y=0.99, xanchor="right", x=0.99, bgcolor="rgba(255,255,255,0.8)")
    )
    
    return fig

In [29]:
DOC_ID_A_CONSULTAR = 3000
df_similares = recomendar_similares_por_topico(DOC_ID_A_CONSULTAR, top_n=10)
display(df_similares)

# Gráfico 1: El texto crudo
fig_semantico = visualizar_constelacion_color(
    DOC_ID_A_CONSULTAR, df_similares, 
    col_x="umap_sem_x", col_y="umap_sem_y", 
    titulo_grafico="Mapa de articulos (Ubicación dada por texto del articulo)"
)
fig_semantico.show()

# Gráfico 2: La receta de temas
fig_tematico = visualizar_constelacion_color(
    DOC_ID_A_CONSULTAR, df_similares, 
    col_x="umap_tem_x", col_y="umap_tem_y", 
    titulo_grafico="Mapa de articulos (Ubicación dada por topico)"
)
fig_tematico.show()

[ID: 3000]
Título:  El jardín mental, ese terreno fértil
Tagline: Testosterona, la obra de Lorena Vega y Cristian Alarcón
Tags:    Laboratorio Periodismo Performàtico; Testosterona

TOP 10 NOTAS MÁS SIMILARES POR PERFIL DE TÓPICOS:



,Rank,doc_id,Similitud_Coseno,Título,Tagline,Etiquetas
0,1,648,0.9163,Las viejas narrativas del presente,La crónica en debate,Rosario; Rosariocronica
1,2,704,0.8680,Un ornitorrinco y cien monedas,Relatoría,Sin etiquetas
2,3,54,0.8602,OVNIS: TE JURO QUE LOS VI,Misterios del universo,Sin etiquetas
3,4,110,0.8565,Un simulacro verdadero,Mario Testino,Sin etiquetas
4,5,36,0.8480,La balada del perro y la muerte,Rosario,Sin etiquetas
5,6,2759,0.8453,Ficciones contra la postverdad,"Markus Gabriel: imaginación, medios y ciencia",Ciencia; Libros
6,7,261,0.8429,Contrapunto,Muestra fotográfica,Sin etiquetas
7,8,3042,0.8428,El museo es noticia,"Arte, ciencia y periodismo: el meteorito y el ...",Rosariocronica
8,9,1289,0.8402,Gente que sí: Mariana Enríquez,Anfibia Podcast - Gente que sí,Sin etiquetas
9,10,969,0.8382,Diario de un bordador,"Hilos, masculinidades y mercado",Sin etiquetas


In [20]:
# Ejecución
fig_temporal_top10 = visualizar_dispersion_temporal_jerarquica(1, top_n=100)
fig_temporal_top10.show()

In [21]:
# ==============================================================================
# EXPORTACIÓN PARA STREAMLIT (Módulo 4: Motor de Recomendación)
# 1. articulos.csv: Fuente única de la verdad con metadatos, tags y UMAPs.
# 2. doc_topic_dist.npy: Matriz de distribución continua de tópicos (N x K).
# ==============================================================================

import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

STREAMLIT_DIR = PROJECT_ROOT / "data" / "streamlit"
STREAMLIT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Matriz matemática pura para Similitud Coseno
np.save(STREAMLIT_DIR / "doc_topic_dist.npy", doc_topic_dist)

# 2. Base Unificada
cols_necesarias = [
    "doc_id", "title_text", "source_url", "tagline", "post_tag", "post_date", 
    "topic", "topic_label", "umap_sem_x", "umap_sem_y", "umap_tem_x", "umap_tem_y"
]

for col in cols_necesarias:
    if col not in df.columns:
        raise KeyError(f"Falta la columna '{col}' en df. Asegurate de haber ejecutado los bloques anteriores de UMAP.")

df_app = df[cols_necesarias].copy()

# Normalizar fecha a string limpio (YYYY-MM-DD)
df_app["post_date"] = pd.to_datetime(df_app["post_date"], errors="coerce").dt.strftime("%Y-%m-%d")

# Exportar base definitiva
df_app.to_csv(STREAMLIT_DIR / "articulos.csv", index=False)

print(f"Exportados correctamente en: {STREAMLIT_DIR}")

Exportados correctamente en: C:\Trabajos\anfibia-unsam\data\streamlit
